In [1]:
from connection_db import connection_with_oracle
from openai import OpenAI
import os 

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 # Dimension for text-embedding-3-small
GENERATION_MODEL = "gpt-5.2"

connection = connection_with_oracle()
cursor = connection.cursor()
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api_key)

Attempting secure MTS Connection
('Connected!',)


In [2]:
# Fetch Available Persona from tables
# We dont want to hardcode the system prompt that we are looking for this kind of user and all so instead we are doing this 

print("Fetching Hiring Personals from Oracle DB")
cursor.execute("SELECT rule_id, agent_persona, evaluation_criteria FROM recruitment_rules")
rules = cursor.fetchall()

active_personas = {}

for r in rules:
    r_id , persona_lob , criteria_lob = r

    # FIX: Check if it's a LOB and read it, otherwise use as string
    persona = persona_lob.read() if hasattr(persona_lob, 'read') else str(persona_lob)
    criteria = criteria_lob.read() if hasattr(criteria_lob, 'read') else str(criteria_lob)

    active_personas[r_id] = {
        "persona": persona,
        "criteria": criteria
    }

    print(f"🆔 ID: {r_id}")
    print(f"👤 Persona: {persona}")
    print(f"📋 Criteria Snippet: {criteria[:80]}...")
    print("-" * 50)

print(f"\n✅ Loaded {len(active_personas)} personas into application memory.")

Fetching Hiring Personals from Oracle DB
🆔 ID: rule_tech_screener
👤 Persona: You are a Senior Technical Architect. Your goal is to find rock-star coders who can hit the ground running.
📋 Criteria Snippet: Look for specific hard skills: Python, Oracle, AWS, CI/CD. Validate years of exp...
--------------------------------------------------
🆔 ID: rule_culture_screener
👤 Persona: You are a Culture Fit Officer. Your goal is to build a collaborative, inclusive team.
📋 Criteria Snippet: Ignore specific tech stack keywords. Focus on words like 'Lead', 'Mentor', 'Comm...
--------------------------------------------------
🆔 ID: rule_junior_scout
👤 Persona: You are a University Recruiter. Your goal is to find diamonds in the rough.
📋 Criteria Snippet: Forgive lack of years of experience. Look for 'Eager to learn', 'Masters degree'...
--------------------------------------------------

✅ Loaded 3 personas into application memory.


In [3]:
from data_ingestion.fetch_and_transform import get_embeddings_batch
import oracledb
def find_candidates_hybrid(user_query, max_salary=1000000, min_experience=0):
    ''' 
        Perfom Hybrid search
        1 :- Vector : Semantically match the user query against the candidate resume
        2 :- SQL : Filter out the candidate who dont meet the salary and experience 
    '''
    print(f"Searching for user query : {user_query}")
    print(f"Constrain : Salary <= ${max_salary} , Experience >= {min_experience}")

    # Vectorize the query using the helper function
    query_vector = get_embeddings_batch([user_query])[0]

    # Prepare a Hybrid Query Search
    # we select the candidate details and the distance score 
    # We use Dot product for similarity(Higher is better)

    sql = """
        SELECT candidate_id, full_name, years_experience, salary_expectation, summary,
               VECTOR_DISTANCE(resume_vector, :v, DOT) as similarity
        FROM candidate_pool
        WHERE salary_expectation <= :max_sal
          AND years_experience >= :min_exp
        ORDER BY similarity DESC
        FETCH FIRST 3 ROWS ONLY
    """

    # Execute
    cursor.setinputsizes(v=oracledb.DB_TYPE_VECTOR)

    cursor.execute(sql, {
        "v": query_vector,
        "max_sal": max_salary,
        "min_exp": min_experience
    })

    results = cursor.fetchall()
    print(f"   -> Found {len(results)} candidates fitting criteria.\n")
    return results

In [4]:
# Unit Test to check the hybrid Query works

test_query = "Experienced Python Developer"
test_salary = 150000
test_exp = 2

print(f"🧪 Testing Engine with: '{test_query}' (Max Salary: ${test_salary:,})")

results = find_candidates_hybrid(test_query, max_salary=test_salary, min_experience=test_exp)

print("--- Test Results ---")
for r in results:
    c_id, name, exp, sal, summary_lob, score = r

     # Handle LOB conversion safely
    summary_text = summary_lob.read() if hasattr(summary_lob, 'read') else str(summary_lob)

    print(f"✅ Found: {name}")
    print(f"   Salary: ${sal:,}")
    print(f"   Match Score: {score:.4f}")
    print("-" * 30)

🧪 Testing Engine with: 'Experienced Python Developer' (Max Salary: $150,000)
Searching for user query : Experienced Python Developer
Constrain : Salary <= $150000 , Experience >= 2
   -> Found 2 candidates fitting criteria.

--- Test Results ---
✅ Found: Jordan L.
   Salary: $85,000
   Match Score: -0.3923
------------------------------
✅ Found: Riley S.
   Salary: $140,000
   Match Score: -0.4706
------------------------------


In [5]:
def generate_hiring_recommendation(user_query, max_salary, min_exp, persona_id="rule_tech_screener"):
    
    # Validation
    if persona_id not in active_personas:
        return f"❌ Error: Persona '{persona_id}' not found. Available: {list(active_personas.keys())}"
    
    # Retrive Context
    persona_data = active_personas[persona_id]
    system_persona = persona_data['persona']
    grading_rubric = persona_data['criteria']

    print(f"🤖 ACTIVATING AGENT: {persona_id}")
    print(f"   Goal: {system_persona}")

    # Hybrid Search
    candidates = find_candidates_hybrid(user_query, max_salary, min_exp)

    if not candidates:
        return "⚠️ No candidates found"
    
    # Augmented the prompt 
    # Turning database raw into a readable textblock 
    context_block = ""

    for c in candidates:
        c_id , name , exp , sal , summary_lob , score = c

        summary = summary_lob.read() if hasattr(summary_lob, 'read') else str(summary_lob)

        context_block += f"""
        --- CANDIDATE PROFILE ---
        ID: {c_id}
        Name: {name}
        Cost: ${sal:,} (Budget: ${max_salary:,})
        Experience: {exp} years
        Resume Summary: {summary}
        (Vector Match Score: {score:.4f})
        -------------------------
        """

    # 5. Construct the Final Prompt
    user_message = f"""
    USER REQUEST: "{user_query}"

    CANDIDATES FOUND (Database Output):
    {context_block}

    INSTRUCTIONS:
    Based on your persona rules below, evaluate these candidates.
    1. Select the BEST fit.
    2. Explain WHY, referencing their specific skills.
    3. If they are over budget or underqualified, mention it as a risk.

    YOUR GRADING RUBRIC:
    {grading_rubric}
    """

    # 6. Generate (Call OpenAI)
    print("🧠 analyzing candidates via GPT...")

    response = client.chat.completions.create(
        model="gpt-5.2",
        messages=[
            {"role": "system", "content": system_persona},
            {"role": "user", "content": user_message}
        ],
        temperature=0.3 # Low temperature for factual evaluation
    )

    return response.choices[0].message.content

In [6]:
# Live Recruitment Simulation

search_query = "We need a Python Backend developer who can lead a team."
max_budget = 160000
min_experience = 4

# Setting up proper agent persona
agent_role = "rule_tech_screener"

print(f"🎬 STARTING SIMULATION")
print(f"   Query: '{search_query}'")
print(f"   Budget: ${max_budget:,}")
print(f"   Agent: {agent_role}\n")
print("=" * 60)

# Execute the pipeline
recommendation = generate_hiring_recommendation(
    user_query=search_query,
    max_salary=max_budget,
    min_exp=min_experience,
    persona_id=agent_role
)

print("\n" + "=" * 60)
print(f"📢 AI RECOMMENDATION ({agent_role})")
print("=" * 60)
print(recommendation)

🎬 STARTING SIMULATION
   Query: 'We need a Python Backend developer who can lead a team.'
   Budget: $160,000
   Agent: rule_tech_screener

🤖 ACTIVATING AGENT: rule_tech_screener
   Goal: You are a Senior Technical Architect. Your goal is to find rock-star coders who can hit the ground running.
Searching for user query : We need a Python Backend developer who can lead a team.
Constrain : Salary <= $160000 , Experience >= 4
   -> Found 2 candidates fitting criteria.

🧠 analyzing candidates via GPT...

📢 AI RECOMMENDATION (rule_tech_screener)
## Best Fit: **CAND_005 — Quinn R.**

### Why Quinn is the best match (Python backend + team lead)
- **Direct Python alignment:** Quinn has a **strong Python background**, which maps to the “Python Backend developer” requirement far better than Riley (who is explicitly a Java backend developer).
- **Proven leadership:** Quinn is an **Engineering Manager** focused on **mentorship, team growth, and aligning engineering goals with business strategy**—t